# ML Laboratuvarı · Modül 4, Ünite 9: Kredi riski

**Prof. Dr. Mehmet Songur** · ML Laboratuvarı, Modül 4

Veri: Tayvan'da bir bankanın 30.000 kredi kartı müşterisi (UCI, Yeh ve Lien 2009, CC BY 4.0). Kurulum hücresi `kredi` adlı tabloyu indirir. Hedef `default`: Ekim 2005'te temerrüt (1). Sütunlar: `limit_bal` (kredi limiti), `sex`, `education`, `marriage`, `age`, `pay_0`, `pay_2` … `pay_6` (Eylül … Nisan ödeme durumu), `bill_amt1` … `bill_amt6` (ekstre tutarı), `pay_amt1` … `pay_amt6` (ödenen tutar). Kodu **Gemini'ye yazdır**, sonra **denetle**.

**Ödeme durumu kodları:** −2 kart kullanılmadı, −1 borcun tamamı ödendi, 0 asgari ödeme yapıldı, 1, 2, … kaç ay gecikildiği. (−2 ve 0 verinin resmî açıklamasında yok; literatürdeki yaygın yorum.)

**Kurallar**
1. **Kurulum** ve **Kontrol** hücrelerini değiştirme; yalnızca çalıştır.
2. Görev hücrelerinde kodu Gemini ile üret; çalıştırmadan önce oku.
3. Kontrol hücresi geçerse bir **tamamlama kodu** verir; onu ML Laboratuvarı'ndaki kutuya yapıştır.

In [ ]:
#@title Kurulum — çalıştır, değiştirme

import base64, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
warnings.filterwarnings("ignore")

kredi = pd.read_csv("https://raw.githubusercontent.com/mehmetsongur/ml-lab/main/data/credit/taiwan_default.csv")
print("kredi:", kredi.shape, "| temerrüt oranı:", round(kredi["default"].mean(), 3))
_K = kredi.copy()
_E, _T = train_test_split(_K, test_size=0.3, random_state=0, stratify=_K["default"])

def _k(s):
    return base64.b64decode(s).decode()

def _maliyet(y, p, e):
    k = p >= e
    return (5 * np.sum(~k & (y == 1)) + np.sum(k & (y == 0))) / len(y)

def _son_adim(model):
    return model.steps[-1][1] if isinstance(model, Pipeline) else model

def kontrol_A(egitim, test, hazirla, model, p_test, auc_test):
    if len(test) != len(_T) or set(test.index) != set(_T.index) or set(egitim.index) != set(_E.index):
        print("❌ Bölme beklenenden farklı: train_test_split(kredi, test_size=0.3, random_state=0, stratify=kredi['default'])."); return
    Xe = hazirla(egitim)
    if {"sex", "marriage"} & set(Xe.columns):
        print("❌ Korunan özellikler (sex, marriage) modelde. Bunları dışarıda bırak."); return
    if "default" in Xe.columns:
        print("❌ Hedef ('default') özelliklerin içinde: hedef sızıntısı."); return
    ham = [k for k in ["pay_0", "pay_2", "pay_3", "pay_4", "pay_5", "pay_6"] if k in Xe.columns]
    if ham:
        print(f"❌ Ödeme durumu kodları sayı olarak duruyor ({', '.join(ham)}). −2 ve −1 kategoridir: gecikme ayı, tam ödeme ve kullanılmama ayrı değişkenler olmalı."); return
    if not isinstance(model, Pipeline) or not any(isinstance(s, StandardScaler) for _, s in model.steps):
        print("❌ Ölçekleme ve lojistik regresyon bir pipeline içinde değil (make_pipeline(StandardScaler(), LogisticRegression(...)))."); return
    son = _son_adim(model)
    if not isinstance(son, LogisticRegression):
        print("❌ Pipeline'ın son adımı LogisticRegression olmalı."); return
    if son.class_weight is not None:
        print("❌ class_weight kullanılmış: olasılıklar şişer ve artık temerrüt olasılığı değildir. class_weight'i kaldır."); return
    if son.C < 1e4:
        print("ℹ️ Bilgi: C küçük; lojistik regresyon cezalı. İstek C=1e6 idi.")
    p = np.asarray(p_test, dtype=float)
    if len(p) != len(test) or len(np.unique(p)) <= 2:
        print("❌ p_test olasılık değil (yalnızca 0/1 içeriyor ya da uzunluğu yanlış). predict yerine predict_proba(...)[:, 1] kullan."); return
    if not np.allclose(p, model.predict_proba(hazirla(test))[:, 1]):
        print("❌ p_test, modelin hazirla(test) için verdiği temerrüt olasılıklarıyla (predict_proba(...)[:, 1]) uyuşmuyor."); return
    a = roc_auc_score(test["default"], p)
    if abs(float(auc_test) - a) > 1e-6:
        print("❌ auc_test, roc_auc_score(test['default'], p_test) ile uyuşmuyor."); return
    if a < 0.75:
        print(f"❌ Test AUC {a:.3f}: beklenenden düşük (≈ 0,76). Kodlar doğru çözülmüş mü? Tutarlar işaretli log ile dönüştürülmüş mü?"); return
    print(f"✅ Tüm kontroller geçti (test AUC ≈ {a:.3f}). Tamamlama kodun: " + _k("TUw0LUtSRC04TldU"))

def kontrol_B(egitim, test, hazirla, model, p_test, esik, maliyet_test):
    izgara = np.round(np.arange(0.05, 0.71, 0.01), 2)
    ye, yt = egitim["default"].values, test["default"].values
    p_cv = cross_val_predict(clone(model), hazirla(egitim), ye, cv=5, method="predict_proba")[:, 1]
    ref = izgara[np.argmin([_maliyet(ye, p_cv, e) for e in izgara])]
    tst = izgara[np.argmin([_maliyet(yt, np.asarray(p_test), e) for e in izgara])]
    e = float(esik)
    if abs(e - ref) > 0.006:
        if abs(e - tst) < 0.006 and abs(tst - ref) > 0.006:
            print(f"❌ Eşik test verisinde seçilmiş ({e:.2f}). Eşiği eğitim verisinde cross_val_predict olasılıklarıyla seç."); return
        print(f"❌ Eşik ({e:.2f}) beklenenden farklı. Maliyetler: kaçan temerrüt 5, reddedilen iyi müşteri 1; ızgara 0,05–0,70; eğitimde cv=5 (karıştırmasız) ile cross_val_predict."); return
    m = _maliyet(yt, np.asarray(p_test), e)
    if abs(float(maliyet_test) - m) > 0.002:
        print("❌ maliyet_test beklenenden farklı: test verisinde, seçilen eşikle başvuru başına ortalama maliyet (5·YN + YP) / n olmalı."); return
    print(f"✅ Tüm kontroller geçti (eşik {e:.2f}, kuramsal 1/6 ≈ 0,17; test maliyeti {m:.3f}, herkesi onayla {_maliyet(yt, np.asarray(p_test), 2):.3f}). Tamamlama kodun: " + _k("TUw0LUVTSy0zSEpW"))

kredi.head(3)

## Görev A — Model

1. `kredi`'yi `train_test_split(kredi, test_size=0.3, random_state=0, stratify=kredi["default"])` ile `egitim` ve `test` olarak böl.
2. `hazirla(df)` adlı bir fonksiyon yaz: `default`, `sex` ve `marriage` sütunlarını at; her `pay_` sütunu için `_gecikme` (0'ın altı 0'a kırpılmış), `_tam` (değer −1 ise 1) ve `_yok` (değer −2 ise 1) değişkenlerini üret ve özgün `pay_` sütununu sil; adında `amt` geçen tutarları işaretli log ile dönüştür: `sign(x)·log(1+|x|)`.
3. `make_pipeline(StandardScaler(), LogisticRegression(C=1e6, max_iter=5000))`'i yalnızca `hazirla(egitim)` ile kur: `model`.
4. Test olasılıkları `p_test = model.predict_proba(hazirla(test))[:, 1]`, test AUC'si `auc_test`.

**Denetim soruları:** sex ve marriage dışarıda mı? `pay_` kodları çözülmüş mü? Ölçekleyici pipeline içinde mi? AUC `predict_proba` ile mi hesaplandı? `class_weight` var mı?

In [ ]:
# Görev A: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol A — çalıştır, değiştirme

In [ ]:
kontrol_A(egitim, test, hazirla, model, p_test, auc_test)

## Görev B — Maliyete göre eşik

Kaçan bir temerrüt (YN) 5, reddedilen iyi bir müşteri (YP) 1 birim maliyetli. Başvuru başına ortalama maliyet: (5·YN + YP) / n.

1. Eğitim verisinde `cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(C=1e6, max_iter=5000)), hazirla(egitim), egitim["default"], cv=5, method="predict_proba")[:, 1]` ile çapraz doğrulama olasılıklarını üret.
2. 0,05'ten 0,70'e 0,01 adımlı bir eşik ızgarasında eğitim maliyetini hesapla; en düşüğünü `esik` olarak seç.
3. Test verisinde, seçilen eşikle maliyeti hesapla: `maliyet_test`. “Herkesi onayla” ve eşik 0,5 ile karşılaştır.

**Denetim soruları:** Eşik test verisiyle mi seçildi? Seçilen eşik kuramsal eşiğe (1/6) yakın mı?

In [ ]:
# Görev B: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol B — çalıştır, değiştirme (birkaç saniye sürebilir)

In [ ]:
kontrol_B(egitim, test, hazirla, model, p_test, esik, maliyet_test)

## Denetim notların

- Gemini özellikleri nasıl hazırladı? Değiştirmen gerekti mi?
- Ölçekleme, AUC ve eşik seçimi nasıl yapıldı?
- Sonucu bir kredi komitesine nasıl anlatırsın? (AUC, kalibrasyon, eşik, maliyet; sınırlılıklar ve adillik)